<a href="https://colab.research.google.com/github/LHEYYUEHUA013/FUNDAI-Lab-PEREZ/blob/main/Lab3_Game_AI_Perez.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3: Game AI Using Minimax with Alpha-Beta pruning

## Fundamentals of Artificial Intelligence

**Name:** Vince Lhey G. Perez
**Course:** BSCS AI
**Section:** 09282 - FUNDAI
**Date:** August 27, 2026

**Selected Game:** Tic-Tac-Toe
**GitHub URL:** https://github.com/LHEYYUEHUA013/FUNDAI-Lab-PEREZ/tree/main

## Description
This laboratory implements a Tic-Tac-Toes AI using Minimax with Alpha-Beta pruning.

The game is playable inside Google Colab using ipywidgets.

In [6]:
import math
import ipywidgets as widgets
from IPython.display import display

class TicTacToeGame:
  X = "X"
  O = "O"
  EMPTY = " "

  def __init__(self):
    self.board = [self.EMPTY] * 9
    self.current_player = self.X

  def available_moves(self):
    return [i for i, value in enumerate(self.board) if value == self.EMPTY]

  def make_move(self, move):
    self.board[move] = self.current_player
    self.current_player = self.O if self.current_player == self.X else self.X

  def undo_move(self, move):
    self.board[move] = self.EMPTY
    self.current_player = self.O if self.current_player == self.X else self.X

  def get_winner(self):
    winning_lines = [
         (0,1,2),
         (3,4,5),
         (6,7,8),
         (0,3,6),
         (1,4,7),
         (2,5,8),
         (0,4,8),
         (2,4,6)
     ]

    for a, b, c in winning_lines:
      if self.board[a] != self.EMPTY and self.board[a] == self.board[b] == self.board[c]:
          return self.board[a]
    return None # Return None if no winner is found

  def is_draw(self):
    return self.get_winner() is None and len(self.available_moves()) == 0

  def is_terminal(self):
    return self.get_winner() is not None or len(self.available_moves()) == 0

  def utility(self):
    winner = self.get_winner()
    if winner == self.X:
      return 1
    elif winner == self.O:
      return -1
    else:
      return 0

  def minimax_alpha_beta(game, alpha=-math.inf, beta=math.inf):
    if game.is_terminal():
      # Consistently return (score, move). In terminal state, no move. This also fixes the recursive call expecting 2 values.
      return game.utility(), None

    # MAX player: X
    if game.current_player == TicTacToeGame.X:
      best_value = -math.inf
      best_move = None

      for move in game.available_moves():
        game.make_move(move)
        # Recursive call expects (value, move)
        value, _ = TicTacToeGame.minimax_alpha_beta(game, alpha, beta)
        game.undo_move(move)

        if value > best_value:
          best_value = value
          best_move = move

        alpha = max(alpha, best_value)
        if alpha >= beta:
          break

      return best_value, best_move
     # MIN player: O
    else:
      best_value = math.inf
      best_move = None

      for move in game.available_moves():
        game.make_move(move)
        # Recursive call expects (value, move)
        value, _ = TicTacToeGame.minimax_alpha_beta(game, alpha, beta)
        game.undo_move(move)

        if value < best_value:
          best_value = value
          best_move = move

        beta = min(beta, best_value)
        if beta >= alpha:
          break

      return best_value, best_move

class TicTacToeUI:
  def __init__(self):
    self.game = TicTacToeGame()

    self.buttons = [
      widgets.Button(
          description=" ",
          layout=widgets.Layout(width="60px", height="60px")
      )
      for _ in range(9)
    ]

    for i in range(9):
      self.buttons[i].on_click(lambda btn, idx=i: self.on_cell_click(idx))

    self.status = widgets.HTML(value="<b>Human X moves first.</b>")
    self.reset_button = widgets.Button(description="Reset", button_style="info")
    self.reset_button.on_click(self.on_reset)

    self.grid = widgets.GridBox(
        children=self.buttons,
        layout=widgets.Layout(
            grid_template_columns="repeat(3, 60px)",
            grid_gap="5px"
        )
    )

    self.widget = widgets.VBox([self.status, self.grid, self.reset_button])
    display(self.widget)

    self.refresh()

  def refresh(self):
    for i, button in enumerate(self.buttons):
      button.description = self.game.board[i]
      button.disabled = self.game.is_terminal() or self.game.board[i] != TicTacToeGame.EMPTY

    if self.game.is_terminal():
      winner = self.game.get_winner()

      if winner:
        self.status.value = f"<b>player {winner} wins! </b>"
      else:
        self.status.value = "<b>Draw!</b>"
    else:
      self.status.value = f"<b>Current player: {self.game.current_player}</b>"
  def on_cell_click(self, index):
    if self.game.board[index] != TicTacToeGame.EMPTY:
      return
    if self.game.is_terminal():
      return
    if self.game.current_player != TicTacToeGame.X:
      return

    self.game.make_move(index)

    if not self.game.is_terminal():
      ai_move = self.get_ai_move() # get_ai_move now returns just the move
      if ai_move is not None:
        self.game.make_move(ai_move)

    self.refresh()

  def get_ai_move(self):
    # Call minimax_alpha_beta as a method of self.game, no extra 'game' argument needed
    # It returns (score, move), we only need the move for get_ai_move
    score, move = self.game.minimax_alpha_beta()
    return move

  def on_reset(self, btn):
    self.game = TicTacToeGame()
    self.refresh()
TicTacToeUI()